### ☀️ 참고 ☀️ 한글깨짐 방지

- 아래 코드셀 3개 실행 필수
- 윈도우, 맥 동일

❗️주의❗️
- Step1. 첫째 셀 실행
- Step2. '런타임 > 세션 다시시작'
- Step3. 이후 두번째 셀 실행

In [ ]:
# 나눔 폰트 설치
# 해당 셀 실행 후 '런타임 > 세션 다시 시작'

%%capture

!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl


# 나눔고딕 폰트 사용 설정 (폰트 설치 후 실행)
plt.rcParams["font.family"]="NanumGothic"

# 마이너스 깨짐 방지 (일반 하이픈(-)을 사용하게 설정)
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 한글 폰트 테스트 그래프

plt.plot([1, 2, 3], [10, -20, 30])
plt.title('한글 폰트 테스트')
plt.xlabel('x축')
plt.ylabel('y축')
plt.grid(True)
plt.show()

# 📘 CH14. 딥러닝 기초 (Keras) — 강의용 노트북

01. 30분 만의 첫 신경망 · 02. 해부실 ① 활성화 함수 · 03. 해부실 ② 학습 엔진 · 04. 손글씨 인식 프로젝트 · 05. 실무 마감 — 콜백과 최종 대결

> 🔄 **개정 안내**: 이 노트북은 "실험실" 방식으로 진행됩니다.
> - **원리·수식 설명은 개념 교안(장표)이 담당**하고, 이 노트북에서는 코드로 개념을 구현하지 않습니다. 대신 **1교시에 실제 동작하는 신경망을 먼저 완성**하고, 2~3교시에 그 모델을 **직접 고장 내보면서** 각 부품이 왜 필요한지 몸으로 확인합니다.
> - 스토리: "AI 스타트업에 입사한 신입 딥러닝 엔지니어의 첫날" — 사수의 방침은 단 하나, "일단 만들고, 뜯어봅시다."
> - 4교시엔 **여러분이 그린 손글씨를 여러분의 모델이 읽는** 프로젝트, 5교시엔 트리 vs 신경망 최종 대결이 기다립니다.
> - TODO 실습과 실습 과제는 정규 수업이 아닌 **수업 후 자습 시간**에 진행합니다.
> - 실습파일: `hr_attrition.csv`(기존 배포 파일 그대로), `mnist_data.npz`, `sample_digit.png` — 모두 사전 배포
> - 🖥️ 라이브 데모와 🎁 보너스는 진도 상황에 따라 건너뛸 수 있는 독립 구성입니다.

---

# 📘 CH14-01. 신경망 만들기 — 일단 만들고, 뜯어봅시다

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 미션 브리핑 → 데이터 준비 → 신경망 조립(Sequential) → 학습(fit) 실시간 관찰 → 첫 예측
> 🔧 **실습파일**: 필요 (`hr_attrition.csv` — 기존 배포 파일 그대로)
> ⏱️ **예상 소요시간**: 40분
> 🎚️ **난이도**: ★★★☆☆

---

## 🤔 먼저 생각해보기

> **AI 스타트업 첫 출근날, 사수가 말합니다 — "딥러닝 이론부터 배우면 3일 걸립니다. 우리는 반대로 갑니다. 지금부터 30분 안에, 트리 모델로 만들었던 퇴사 예측을 '진짜 신경망'으로 완성할 겁니다. 코드 몇 줄은 지금 이해가 안 될 텐데, 정상입니다 — 오후 내내 그 코드를 하나씩 뜯어볼 거니까요."**

오늘의 전략은 **선(先)완성, 후(後)해부**입니다. 이번 교시에 만드는 모델이 오늘 하루의 "실험 대상"이 되고, 2~3교시에 이 모델을 일부러 고장 내보면서 활성화 함수·손실 함수·옵티마이저가 왜 필요한지 확인합니다.

---

## 📖 만들기

### 1) 데이터 준비

딥러닝이라고 데이터 준비가 달라지지 않습니다. 단 하나, **스케일링**만 새로 추가됩니다 — 신경망은 입력 숫자들의 크기가 제각각이면 학습이 느리고 불안정해지기 때문입니다. (왜 그런지는 개념 교안에서 — 지금은 "신경망 앞에는 스케일링"만 기억하세요.)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 퇴사 예측 데이터 (트리 모델 때와 동일한 전처리)
df = pd.read_csv("hr_attrition.csv")
df = df.drop(columns=["EmployeeNumber", "EmployeeCount", "StandardHours", "Over18"], errors="ignore")
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

X = pd.get_dummies(df.drop(columns=["Attrition"]), drop_first=True)
y = df["Attrition"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 새로 추가되는 단 한 단계: 스케일링 (신경망의 필수 전처리)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled.shape)   # (1176, 44) — 직원 1,176명 × 특성 44개
```

</details>

In [ ]:
# 코드 입력

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 퇴사 예측 데이터 (트리 모델 때와 동일한 전처리)

df = pd.read_csv("hr_attrition.csv")

# 불필요 컬럼 삭제 ("EmployeeNumber", "EmployeeCount", "StandardHours", "Over18")
# errors="ignore" 옵션 추가

df = df.drop(columns=["EmployeeNumber", "EmployeeCount", "StandardHours", "Over18"], errors="ignore")

# 목표변수 인코딩 (레이블 인코딩)
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

# 범주형 변수 인코딩 (원핫 인코딩) + 데이터 분리
X = pd.get_dummies(df.drop(columns=["Attrition"]), drop_first=True)
y = df["Attrition"]

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 새로 추가되는 단 한 단계: 스케일링 (신경망의 필수 전처리)





print(X_train_scaled.shape)
# (1176, 44) — 직원 1,176명 × 특성 44개

### 2) 신경망 조립 — 레고 블록처럼

지금부터 딱 4줄로 신경망을 조립합니다. 각 줄의 의미는 아래 표에 요약해뒀고, **"왜 relu인지, 왜 sigmoid인지"는 다음 교시의 실험 주제**입니다 — 지금은 조립만 하세요.

| 코드 | 지금 알아둘 것 |
|---|---|
| `keras.Input(shape=(44,))` | 입력 구멍: 특성 44개가 들어감 |
| `Dense(16, activation="relu")` | 뉴런 16개짜리 층 (relu = 2교시 실험 대상) |
| `Dense(8, activation="relu")` | 뉴런 8개짜리 층 하나 더 |
| `Dense(1, activation="sigmoid")` | 출구: "퇴사 확률" 숫자 하나 (sigmoid = 2교시 실험 대상) |

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

# 학습 방법 설정: "adam 방식으로, binary_crossentropy 기준으로 배우세요" (둘 다 3교시 실험 대상)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.summary()   # 방금 조립한 구조 확인 — Param = 모델이 스스로 조정할 숫자들의 개수
```

</details>

In [ ]:
# 코드 입력





### 3) 학습 — 이 화면을 잘 보세요

`fit()`을 실행하면 로그가 30줄 올라옵니다. **관전 포인트는 딱 하나: `loss` 숫자가 점점 줄어드는 것.** 이것이 "학습"의 정체입니다 — 모델이 865개의 숫자(Param)를 조금씩 조정하면서 오답(loss)을 줄여가는 과정이 실시간으로 보이는 겁니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
history = model.fit(
    X_train_scaled, y_train,
    epochs=30,                 # 전체 데이터를 30바퀴 반복 학습
    batch_size=32,              # 한 번에 32명씩 묶어서
    validation_split=0.2,       # 훈련 데이터의 20%는 "모의고사"용으로 떼어둠
    verbose=1
)
```

</details>

In [ ]:
# 코드 입력






> 💡 로그의 네 숫자 읽는 법: `loss`(훈련 오답 정도) / `accuracy`(훈련 정확도) / `val_loss`·`val_accuracy`(모의고사 성적). **훈련 성적은 계속 오르는데 모의고사 성적이 멈추면?**

### 4) 첫 예측 — 신경망이 답을 내놓는 순간

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 시험(테스트 데이터) 성적
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"테스트 정확도: {test_acc:.4f}")

# 직원 5명의 퇴사 확률 — 신경망의 출력은 0~1 사이의 "확률"
probs = model.predict(X_test_scaled[:5], verbose=0)
for i, p in enumerate(probs):
    print(f"직원 {i+1}: 퇴사 확률 {p[0]*100:.1f}%")
```

</details>

In [ ]:
# 시험(테스트 데이터) 성적
# evaluate()
# 모델 정의할 때 loss > binary_crossentropy, metrics > accuracy로 정함







for i, p in enumerate(probs):
    print(f"직원 {i+1}: 퇴사 확률 {p[0]*100:.1f}%")

🎉 **완성입니다.** 트리 모델이 아닌 "진짜 신경망"이 여러분 손에서 학습을 마쳤고, 확률까지 내놓았습니다. 그런데 지금 코드에는 정체불명의 단어가 4개 있습니다 — `relu`, `sigmoid`, `adam`, `binary_crossentropy`. 지금부터 오후 내내, 이것들을 하나씩 **빼고, 바꾸고, 부수면서** 왜 필요한지 확인합니다.

---

## 🚀 실무에서는?

> 🚀 실무 딥러닝도 정확히 이 4단계입니다 — 데이터 준비 → 조립 → compile → fit. 모델이 아무리 거대해져도(GPT도!) 이 골격은 같습니다. 실무자들도 새 문제를 만나면 "일단 작은 모델을 빠르게 완성"하고 나서 개선을 반복합니다 — 오늘 여러분이 한 방식이 실무의 표준 루틴입니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ 스케일링 없이 `fit()`에 원본 데이터를 넣으면 학습이 매우 느리거나 불안정해집니다 — 신경망 앞에는 항상 스케일링.

> ⚠️ `scaler.fit_transform()`은 훈련 데이터에만, 테스트 데이터에는 `transform()`만 — 시험 문제로 공부하면 반칙입니다.

---

## 🧩 Check Point

**Q1.** 모델을 실제로 학습시키는 메서드는? ① compile() ② summary() ③ fit() ④ predict()
<details><summary>정답</summary>③ fit()</details>

**Q2.** 학습 로그에서 "모델이 배우고 있다"를 보여주는 핵심 신호는? ① epoch 숫자가 커짐 ② loss가 점점 줄어듦 ③ 로그가 빨리 올라감
<details><summary>정답</summary>② loss가 점점 줄어듦</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. (자습 시간에 진행합니다.)

In [ ]:
# TODO 1 (구조 실험): 은닉층을 (32, 16) 뉴런으로 바꾼 모델을 만들어 30 epoch 학습시키고 테스트 정확도를 비교하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
model_big = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_big.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_big.fit(X_train_scaled, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)
print(model_big.evaluate(X_test_scaled, y_test, verbose=0))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (epoch 실험): epochs를 5로 줄여 학습시키고, 30 epoch 모델과 테스트 정확도를 비교하세요 (덜 배운 모델은 얼마나 못할까?)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
model_short = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_short.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_short.fit(X_train_scaled, y_train, epochs=5, batch_size=32, validation_split=0.2, verbose=0)
print(model_short.evaluate(X_test_scaled, y_test, verbose=0))
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

> 📝 자습 시간에 진행합니다.

In [ ]:
# 과제 1: batch_size를 8, 32, 128로 바꿔가며 각각 학습시키고, 학습 시간과 최종 검증 정확도를 비교하세요

# 과제 2: predict()로 테스트 데이터 전체의 퇴사 확률을 구한 뒤, 확률이 가장 높은 상위 5명을 찾아보세요
#         (트리 모델 때 했던 "위험군 상위 N명 추리기"가 신경망에서도 그대로 됩니다)


**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
import time
for bs in [8, 32, 128]:
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    start = time.time()
    h = m.fit(X_train_scaled, y_train, epochs=10, batch_size=bs, validation_split=0.2, verbose=0)
    print(f"batch_size={bs}: {time.time()-start:.1f}초, 검증 정확도 {h.history['val_accuracy'][-1]:.4f}")
```

</details>

In [ ]:
# 과제 1 예시 정답






<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
all_probs = model.predict(X_test_scaled, verbose=0).flatten()
top5 = pd.Series(all_probs, index=X_test.index).sort_values(ascending=False).head(5)
print("퇴사 위험 상위 5명:")
print((top5 * 100).round(1))
```

</details>

In [ ]:
# 과제 2 예시 정답






---

## 📌 핵심 정리

- 딥러닝 4단계: 데이터 준비(+스케일링) → Sequential 조립 → compile → fit
- fit 로그의 loss가 줄어드는 것 = 학습이 되고 있다는 신호
- 신경망의 출력(sigmoid)은 0~1 사이의 확률
- 오늘 만든 이 모델이 오후 실험의 대상 — relu, sigmoid, adam, binary_crossentropy를 이제 해부합니다

---

# 📘 CH14-02. 해부실 ① — 활성화 함수: 빼면 어떻게 되는지 직접 봅시다

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 브라우저 실험(TF Playground) → 실험 A: relu를 빼면? → 실험 B: sigmoid를 빼면? → 활성화 함수 치트시트
> 🔧 **실습파일**: 필요 (1교시 데이터 이어서 사용)
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★☆☆
> 📊 **개념 교안**: 활성화 함수의 수식·그래프·종류는 장표에서 다룹니다 — 이 노트북은 "직접 빼보는 실험"만 합니다.

---

## 🤔 먼저 생각해보기

> **사수의 첫 해부 지시 — "1교시 코드에서 `relu`라는 단어, 왜 있는지 궁금하죠? 설명은 장표에서 봤을 테니, 여기선 다르게 갑니다. 그냥 빼버리세요. 그리고 모델이 어떻게 되는지 직접 보세요."**

활성화 함수의 역할은 한 문장입니다 — **"구부리는 힘".** 이게 없으면 신경망은 층을 아무리 쌓아도 직선 하나짜리 모델과 같아집니다. 말로는 와닿지 않으니, 두 가지 방법으로 직접 확인합니다: ① 브라우저에서 눈으로, ② 우리 모델에서 숫자로.

---

## 🖥️ 실험 0 — 브라우저 신경망 실험실

**🔗 접속**: [TensorFlow Playground](https://playground.tensorflow.org/) — 설치 없이 브라우저에서 신경망을 조종하는 구글의 실험 도구

**실험 순서 (강사 시연 후 각자 도전)**
1. 오른쪽 데이터셋에서 **나선형(spiral)** 선택 — 직선으로는 절대 못 나누는 "최종 보스" 데이터입니다.
2. Activation을 **Linear**로 바꾸고 학습(▶) — 층과 뉴런을 아무리 늘려도 **절대 못 풉니다.** 배경의 경계선이 끝까지 직선인 것을 확인하세요.
3. Activation을 **ReLU**로 바꾸고 재도전 — 경계선이 **구부러지기 시작**하며 나선을 감싸는 순간이 옵니다.

**✏️ 자유 체험**: 각자 층/뉴런 조합으로 나선 정복에 도전해보세요.  — 방금 눈으로 본 그 차이를, 이제 우리 퇴사 예측 모델에서 숫자로 재현합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 실험 A: 같은 모델에서 "구부리는 힘"만 제거 — activation="relu" vs "linear"
def build_and_train(activation):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation=activation),
        layers.Dense(8, activation=activation),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=0)
    return m.evaluate(X_test_scaled, y_test, verbose=0)[1]

acc_relu = build_and_train("relu")
acc_linear = build_and_train("linear")

print(pd.DataFrame({
    "모델": ["relu (구부리는 힘 있음)", "linear (직선만 가능)"],
    "테스트 정확도": [round(acc_relu, 4), round(acc_linear, 4)],
}))
```

</details>

In [ ]:
# 실험 A: 같은 모델에서 "구부리는 힘"만 제거 — activation="relu" vs "linear"









print(pd.DataFrame({
    "모델": ["relu (구부리는 힘 있음)", "linear (직선만 가능)"],
    "테스트 정확도": [round(acc_relu, 4), round(acc_linear, 4)],
}))


> 💡 이 데이터에서는 차이가 작을 수도, 클 수도 있습니다 — 중요한 관찰은 이것입니다: **linear 모델은 층이 3개여도 사실상 로지스틱 회귀 1층과 같은 표현력**밖에 없습니다. Playground의 나선처럼 데이터가 복잡할수록 이 차이는 극적으로 벌어집니다. "층을 쌓는 의미"를 만들어주는 것이 활성화 함수입니다.

> ReLU가 항상 정확도가 높은 것은 아닙니다. 데이터가 단순하면 Linear만으로도 충분합니다. 하지만 데이터가 복잡해질수록 Linear는 아무리 층을 많이 쌓아도 한계가 있고, ReLU 같은 활성화 함수가 있어야 신경망이 복잡한 패턴을 학습할 수 있습니다.

### 실험 B — 출구의 sigmoid를 빼면?

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 실험 B: 출력층의 sigmoid 제거 — "확률"이 어떻게 망가지는지 관찰
model_no_sig = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1),                       # sigmoid 없음 — 출력이 아무 숫자나 됨
])
model_no_sig.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_no_sig.fit(X_train_scaled, y_train, epochs=10, batch_size=32, verbose=0)

print("sigmoid 있음 →", model.predict(X_test_scaled[:3], verbose=0).flatten().round(3), "(전부 0~1 사이: 확률)")
print("sigmoid 없음 →", model_no_sig.predict(X_test_scaled[:3], verbose=0).flatten().round(3), "(범위 제멋대로: 확률이 아님)")
```

</details>

In [ ]:
# 실험 B: 출력층의 sigmoid 제거 — "확률"이 어떻게 망가지는지 관찰









print("sigmoid 있음 →", model.predict(X_test_scaled[:3], verbose=0).flatten().round(3), "(전부 0~1 사이: 확률)")
print("sigmoid 없음 →", model_no_sig.predict(X_test_scaled[:3], verbose=0).flatten().round(3), "(범위 제멋대로: 확률이 아님)")


> 💡 sigmoid는 어떤 숫자든 0~1 사이로 눌러 담는 "출구 필터"입니다. 이게 없으면 "퇴사 확률 -2.3" 같은 해석 불가능한 출력이 나옵니다. 그래서 출력층의 활성화 함수는 **문제 유형이 결정**합니다:

## 📋 활성화 함수 치트시트 (이것만 기억하면 됩니다)

| 위치 | 무엇을 | 이유 |
|---|---|---|
| 은닉층 | **relu** (고민 없이 기본값) | 구부리는 힘 + 학습이 빠름 |
| 출력층 — 이진분류 | **sigmoid** (뉴런 1개) | 0~1 확률 하나 |
| 출력층 — 다중분류 | **softmax** (뉴런 = 클래스 수) | 클래스별 확률, 합쳐서 1 (→ 4교시에 직접 사용) |
| 출력층 — 회귀 | 없음 | 숫자를 그대로 출력해야 하므로 |

---

## ⚠️ 자주 하는 실수

> ⚠️ 이진분류인데 출력층을 (뉴런 2개 + softmax)로, 다중분류인데 (뉴런 1개 + sigmoid)로 조립하는 혼동이 가장 흔합니다 — 위 치트시트로 항상 확인하세요.

> ⚠️ 은닉층에 활성화 함수를 아예 빼먹으면(=linear) 층을 쌓는 의미가 사라집니다.

---

## 🧩 Check Point

**Q1.** 은닉층에 활성화 함수가 없으면 어떻게 되나요? ① 학습이 빨라진다 ② 층을 쌓아도 직선 모델과 같아진다 ③ 에러가 난다
<details><summary>정답</summary>② 층을 쌓아도 직선 모델과 같아진다 (실험 A + Playground에서 확인)</details>

**Q2.** 10개 클래스를 분류하는 모델의 출력층 구성은? ① 뉴런 1개 + sigmoid ② 뉴런 10개 + softmax ③ 뉴런 10개 + relu
<details><summary>정답</summary>② 뉴런 10개 + softmax</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. (자습 시간에 진행합니다.)

In [ ]:
# TODO 1 (tanh 실험): 은닉층 활성화를 "tanh"로 바꾼 모델을 학습시키고 relu·linear와 정확도를 비교하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
acc_tanh = build_and_train("tanh")
print(pd.DataFrame({
    "활성화": ["relu", "tanh", "linear"],
    "테스트 정확도": [round(acc_relu, 4), round(acc_tanh, 4), round(acc_linear, 4)],
}))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (Playground 미션)
# playground.tensorflow.org에서 나선형 데이터를 정복한 구성(층 수, 뉴런 수, 활성화)을 주석으로 기록하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답 (구성은 사람마다 다를 수 있습니다)
# 성공 예: 은닉층 2개 (8, 8 뉴런), Activation=ReLU, Learning rate=0.03 → 약 300 epoch에서 정복
# 관찰: Linear로는 어떤 구성으로도 불가능, ReLU/Tanh로 바꾸는 순간 경계선이 구부러짐
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

> 📝 자습 시간에 진행합니다.

In [ ]:
# 과제 1: 은닉층 개수를 1개/2개/4개로 바꿔가며 (활성화는 relu 고정) 테스트 정확도를 비교하세요
#         — "층이 많을수록 무조건 좋은가?"에 대한 여러분만의 답을 주석으로 적어보세요

# 과제 2: 실험 B의 sigmoid 없는 모델로 predict한 값에 시그모이드 공식(1/(1+e^-x))을 직접 적용하면
#         0~1 사이 값이 되는지 확인하세요 (힌트: np.exp 사용)


**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
def build_depth(n_layers):
    layer_list = [keras.Input(shape=(44,))]
    for _ in range(n_layers):
        layer_list.append(layers.Dense(16, activation="relu"))
    layer_list.append(layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(layer_list)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=0)
    return m.evaluate(X_test_scaled, y_test, verbose=0)[1]

for n in [1, 2, 4]:
    print(f"은닉층 {n}개: 테스트 정확도 {build_depth(n):.4f}")
# 관찰: 이 데이터 규모에서는 층을 늘려도 거의 좋아지지 않거나 오히려 나빠짐 — 층이 많다고 무조건 좋은 게 아님
```

</details>

In [ ]:
# 과제 1 예시 정답






# 관찰: 이 데이터 규모에서는 층을 늘려도 거의 좋아지지 않거나 오히려 나빠짐 — 층이 많다고 무조건 좋은 게 아님


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
import numpy as np
raw = model_no_sig.predict(X_test_scaled[:3], verbose=0).flatten()
manual_sigmoid = 1 / (1 + np.exp(-raw))
print("원본 출력:", raw.round(3))
print("시그모이드 적용:", manual_sigmoid.round(3), "→ 전부 0~1 사이가 됨")
```

</details>

In [ ]:
# 과제 2 예시 정답





---

## 📌 핵심 정리

- 활성화 함수 = "구부리는 힘" — 없으면 층을 쌓아도 직선 모델 (실험 A·Playground로 확인)
- 출력층 활성화는 문제 유형이 결정: 이진=sigmoid / 다중=softmax / 회귀=없음
- 은닉층은 고민 없이 relu가 기본값
- 수식과 그래프는 개념 교안 참고 — 노트북에서는 "빼보고 확인"이 핵심

---

# 📘 CH14-03. 해부실 ② — 학습 엔진: 일부러 부숴봅시다

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 실험 A: 옵티마이저 대결(SGD vs Adam) → 실험 B: 학습률 폭주 실험 → 실험 C: 손실 함수 잘못 끼우기 → 학습 엔진 치트시트
> 🔧 **실습파일**: 필요 (1교시 데이터 이어서 사용)
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★★☆
> 📊 **개념 교안**: 경사하강법·손실 함수·옵티마이저의 원리는 장표에서 다룹니다 — 이 노트북은 "부숴보는 실험"만 합니다.

---

## 🤔 먼저 생각해보기

> **사수의 두 번째 지시 — "compile() 안의 `adam`과 `binary_crossentropy`, 이게 학습 엔진입니다. 엔진이 뭘 하는지 이해하는 최고의 방법은? 일부러 고장 내보는 겁니다. 느린 엔진으로 바꿔보고, 액셀을 끝까지 밟아서 폭주도 시켜보고, 엉뚱한 부품도 끼워보세요."**

학습의 원리는 한 문장입니다 — **"loss(오답 점수)가 줄어드는 방향으로 조금씩 이동한다."** 옵티마이저는 "어떻게 이동할지", 학습률은 "한 걸음의 보폭", 손실 함수는 "오답 점수를 매기는 채점 기준"입니다. 전부 그래프로 직접 확인합니다.

---

## 📖 실험

### 실험 A — 옵티마이저 대결: 구형 엔진(SGD) vs 신형 엔진(Adam)

같은 모델, 같은 데이터, 옵티마이저 **한 단어만** 다릅니다. loss가 떨어지는 속도를 겹쳐 그려봅니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt

def train_with(optimizer_name, epochs=30):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=optimizer_name, loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=32, verbose=0)
    return h.history["loss"]

loss_sgd = train_with("sgd")
loss_adam = train_with("adam")

plt.figure(figsize=(8, 4))
plt.plot(loss_sgd, label="SGD (구형 엔진)", marker="o", markersize=3)
plt.plot(loss_adam, label="Adam (신형 엔진)", marker="o", markersize=3)
plt.xlabel("Epoch")
plt.ylabel("Loss (오답 점수)")
plt.title("옵티마이저 대결 — 같은 모델, 엔진만 교체")
plt.legend()
plt.show()
```

</details>

In [ ]:
# 코드 입력

import matplotlib.pyplot as plt










plt.figure(figsize=(8, 4))

plt.plot(loss_sgd, label="SGD (구형 엔진)", marker="o", markersize=3)
plt.plot(loss_adam, label="Adam (신형 엔진)", marker="o", markersize=3)

plt.xlabel("Epoch")
plt.ylabel("Loss (오답 점수)")
plt.title("옵티마이저 대결 — 같은 모델, 엔진만 교체")

plt.legend()

plt.show()


> 💡 Adam이 훨씬 빠르게 loss를 떨어뜨리는 것이 보이나요? Adam은 "보폭을 상황에 맞게 알아서 조절하는 똑똑한 SGD"입니다. **실무 결론: 특별한 이유가 없으면 옵티마이저는 그냥 adam.**

### 실험 B — 학습률 폭주 실험: 액셀을 끝까지 밟으면

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 학습률(learning rate) = 한 걸음의 보폭. 너무 작으면? 너무 크면? 세 대를 동시에 달리게 합니다
def train_with_lr(lr, epochs=30):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=32, verbose=0)
    return h.history["loss"]

plt.figure(figsize=(8, 4))
for lr, label in [(0.0001, "0.0001 (너무 소심함)"), (0.001, "0.001 (적당함 — 기본값)"), (1.0, "1.0 (폭주)")]:
    plt.plot(train_with_lr(lr), label=f"lr={label}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("학습률 실험 — 보폭이 학습을 좌우한다")
plt.legend()
plt.show()
```

</details>

In [ ]:
# 학습률(learning rate)
# 한 걸음의 보폭. 너무 작으면? 너무 크면? 세 대를 동시에 달리게 합니다


def train_with_lr(lr, epochs=30):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=32, verbose=0)
    return h.history["loss"]


plt.figure(figsize=(8, 4))


for lr, label in [(0.0001, "0.0001 (너무 소심함)"), (0.001, "0.001 (적당함 — 기본값)"), (1.0, "1.0 (폭주)")]:
    plt.plot(train_with_lr(lr), label=f"lr={label}")


plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("학습률 실험 — 보폭이 학습을 좌우한다")
plt.legend()
plt.show()


> 💡 세 곡선의 표정이 전부 다릅니다 — **0.0001**은 내려가긴 하는데 답답하게 느리고, **0.001**은 매끄럽게 수렴하고, **1.0**은 loss가 요동치거나 아예 커집니다(폭주). 보폭이 너무 크면 "정답 근처를 계속 뛰어넘어 다니는" 상태가 됩니다. Adam의 기본값 0.001이 괜히 기본값이 아닙니다.

### 실험 C — 채점 기준 잘못 끼우기: 분류 문제에 회귀용 손실을?

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 손실 함수 = 채점 기준. 분류 문제에 회귀용 채점표(mse)를 끼우면 어떻게 되는지 비교
def train_with_loss(loss_name, epochs=30):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss=loss_name, metrics=["accuracy"])
    m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=32, verbose=0)
    return m.evaluate(X_test_scaled, y_test, verbose=0)[1]

acc_ce = train_with_loss("binary_crossentropy")
acc_mse = train_with_loss("mse")

print(pd.DataFrame({
    "손실 함수": ["binary_crossentropy (분류 전용 채점표)", "mse (회귀용 채점표를 잘못 끼움)"],
    "테스트 정확도": [round(acc_ce, 4), round(acc_mse, 4)],
}))
```

</details>

In [ ]:
# 손실 함수
# 채점 기준. 분류 문제에 회귀용 채점표(mse)를 끼우면 어떻게 되는지 비교

def train_with_loss(loss_name, epochs=30):
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    m.compile(optimizer="adam", loss=loss_name, metrics=["accuracy"])
    m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=32, verbose=0)
    return m.evaluate(X_test_scaled, y_test, verbose=0)[1]

acc_ce = train_with_loss("binary_crossentropy")
acc_mse = train_with_loss("mse")

print(pd.DataFrame({
    "손실 함수": ["binary_crossentropy (분류 전용 채점표)", "mse (회귀용 채점표를 잘못 끼움)"],
    "테스트 정확도": [round(acc_ce, 4), round(acc_mse, 4)],
}))


> 💡 mse로도 "돌아는 갑니다" — 그게 더 무섭습니다. 에러 없이 조용히 성능만 손해 봅니다. cross-entropy는 "확신을 갖고 틀렸을 때" 훨씬 큰 벌점을 주는 분류 전용 채점표라서, 분류 문제에서는 더 좋은 방향으로 모델을 밀어줍니다. **문제 유형과 채점표는 반드시 세트.**

## 📋 학습 엔진 치트시트

| 부품 | 기본 선택 | 언제 바꾸나 |
|---|---|---|
| 옵티마이저 | **adam** | 거의 안 바꿈 (실무 표준) |
| 학습률 | **0.001** (adam 기본값) | loss가 폭주하면 ↓ / 너무 느리면 ↑ |
| 손실 — 이진분류 | **binary_crossentropy** | (sigmoid 출력과 세트) |
| 손실 — 다중분류 | **sparse_categorical_crossentropy** | (softmax 출력과 세트)|
| 손실 — 회귀 | **mse** | 이상치 많으면 mae |

---

## ⚠️ 자주 하는 실수

> ⚠️ 분류에 mse, 회귀에 crossentropy — 잘못 끼워도 에러가 안 나는 경우가 많아서 더 위험합니다. 조용히 성능만 나빠집니다.

> ⚠️ loss가 요동치거나 커지면(폭주) 모델 탓이 아니라 학습률이 너무 큰 것부터 의심하세요.

---

## 🧩 Check Point

**Q1.** loss가 줄지 않고 요동치며 커집니다. 가장 먼저 의심할 것은? ① 데이터 오류 ② 학습률이 너무 큼 ③ 층이 부족함
<details><summary>정답</summary>② 학습률이 너무 큼 (실험 B의 lr=1.0 곡선)</details>

**Q2.** 다중분류 모델의 (출력층, 손실 함수) 올바른 세트는? ① sigmoid + mse ② softmax + sparse_categorical_crossentropy ③ relu + binary_crossentropy
<details><summary>정답</summary>② softmax + sparse_categorical_crossentropy</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. (자습 시간에 진행합니다.)

In [ ]:
# TODO 1 (옵티마이저 추가 대결): "rmsprop"도 참전시켜 SGD·Adam과 loss 곡선을 3파전으로 비교하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
loss_rms = train_with("rmsprop")
plt.plot(loss_sgd, label="SGD")
plt.plot(loss_adam, label="Adam")
plt.plot(loss_rms, label="RMSprop")
plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.show()
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (폭주 경계 찾기): lr을 0.01, 0.1, 0.5로 바꿔가며 어느 지점부터 loss가 불안정해지는지 찾아보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for lr in [0.01, 0.1, 0.5]:
    final_loss = train_with_lr(lr, epochs=15)[-1]
    print(f"lr={lr}: 최종 loss {final_loss:.4f}")
# 대체로 0.1 근처부터 불안정해지기 시작 (데이터·모델에 따라 다름)
```

</details>

In [ ]:
# 코드 입력






# 대체로 0.1 근처부터 불안정해지기 시작 (데이터·모델에 따라 다름)

---

## 📝 실습 과제

> 📝 자습 시간에 진행합니다.

In [ ]:
# 과제 1: SGD에 learning_rate를 0.01, 0.1로 각각 지정해 학습시키고,
#         "느린 엔진도 보폭을 잘 잡으면 얼마나 따라잡는지" Adam(기본값)과 최종 loss를 비교하세요
#         (힌트: keras.optimizers.SGD(learning_rate=...))

# 과제 2: 실험 C를 뒤집어보세요 — 만약 "회귀 문제"라면 어떤 손실을 써야 할까요?
#         자전거 수요 데이터(bike_sharing_hourly.csv)로 (출력층 활성화 없음 + mse) 회귀 신경망을 만들어
#         RMSE를 계산해보세요. 트리 모델과 비교하면 어느 쪽이 나은가요?


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
for lr in [0.01, 0.1]:
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=0)
    print(f"SGD lr={lr}: 최종 loss {h.history['loss'][-1]:.4f}")
print(f"Adam 기본값: 최종 loss {loss_adam[-1]:.4f}")
```

</details>

In [ ]:
# 과제 1 답안





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
import numpy as np
bike = pd.read_csv("bike_sharing_hourly.csv")
bike_enc = pd.get_dummies(bike, columns=["season", "weather"], drop_first=True)
Xb = bike_enc.drop(columns=["count"]).astype(float)
yb = bike_enc["count"]
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=42)
sc = StandardScaler()
Xb_tr_s, Xb_te_s = sc.fit_transform(Xb_tr), sc.transform(Xb_te)

reg = keras.Sequential([
    keras.Input(shape=(Xb_tr_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),                        # 회귀: 출력층 활성화 없음
])
reg.compile(optimizer="adam", loss="mse")
reg.fit(Xb_tr_s, yb_tr, epochs=30, batch_size=64, verbose=0)
pred = reg.predict(Xb_te_s, verbose=0).flatten()
rmse = np.sqrt(((yb_te - pred) ** 2).mean())
print(f"신경망 회귀 RMSE: {rmse:.1f}")
# 참고: 같은 데이터의 트리 모델(RandomForest 등) RMSE와 비교해보면, 정형 데이터에선 트리가 대체로 우세
```

</details>

In [ ]:
# 과제 2 답안






# 참고: 같은 데이터의 트리 모델(RandomForest 등) RMSE와 비교해보면, 정형 데이터에선 트리가 대체로 우세


---

## 📌 핵심 정리

- 학습 = "loss가 줄어드는 방향으로 조금씩 이동" — 옵티마이저(어떻게), 학습률(보폭), 손실(채점 기준)
- 옵티마이저는 adam, 학습률은 0.001이 검증된 기본값 (실험 A·B로 이유 확인)
- 문제 유형과 손실 함수는 반드시 세트 — 잘못 끼워도 에러가 안 나서 더 위험 (실험 C)
- 원리와 수식은 개념 교안 참고 — 노트북에서는 "부숴보고 확인"이 핵심

---

# 🖥️ 쉬는시간 데모 — 신경망과 스무고개

> ⏭️ 진도 상황에 따라 건너뛸 수 있는 독립 코너입니다.

**🔗 접속**: [Quick, Draw! (Google)](https://quickdraw.withgoogle.com/) — 여러분이 그림을 그리면 신경망이 실시간으로 알아맞히는 게임입니다. 6문제, 2분이면 끝나요. 방금까지 배운 "학습된 신경망의 예측"이 게임이 되면 이런 모습입니다.

> 💡 이 게임으로 모인 낙서 5천만 장은 실제로 공개 데이터셋이 되어 전 세계 연구에 쓰이고 있습니다 — 여러분이 방금 그린 그림도 누군가의 학습 데이터가 됩니다.

---

# 📘 CH14-04. 프로젝트 — 내 손글씨를 읽는 AI 만들기

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 미션 브리핑(MNIST) → 데이터 구경 → 다중분류 모델 조립(softmax) → 학습·평가 → 헷갈린 글씨 구경 → **내가 그린 숫자 인식시키기**
> 🔧 **실습파일**: 필요 (`mnist_data.npz`, `sample_digit.png` — 사전 배포)
> ⏱️ **예상 소요시간**: 50분
> 🎚️ **난이도**: ★★★★☆

---

## 🤔 먼저 생각해보기

> **사수 — "해부 끝났으면 진짜 프로젝트 갑시다. 1990년대 미국 우체국은 사람이 우편번호를 일일이 읽어 분류했습니다. 이걸 자동화한 것이 딥러닝의 첫 상업적 성공 사례예요. 오늘 그 문제를 여러분이 직접 풉니다 — 그리고 마지막엔, 여러분이 마우스로 쓴 글씨를 여러분의 모델이 읽게 만들 겁니다."**

손글씨 숫자 7만 장(MNIST)으로 **10개 클래스 분류** 모델을 만듭니다. 오전에 배운 치트시트가 그대로 적용됩니다 — 출력층은 뉴런 10개 + **softmax**, 손실은 **sparse_categorical_crossentropy**. 새 개념이 아니라, 치트시트의 첫 실전 사용입니다.

---

## 📖 프로젝트

### 1) 데이터 구경 — 사람들이 실제로 쓴 글씨 25장

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import numpy as np

# 사전 배포된 mnist_data.npz를 노트북과 같은 폴더에 두고 실행하세요
mnist = np.load("mnist_data.npz")
x_train_img, y_train_m = mnist["x_train"], mnist["y_train"]
x_test_img, y_test_m = mnist["x_test"], mnist["y_test"]
print(x_train_img.shape)   # (60000, 28, 28) — 28×28 픽셀 이미지 6만 장

fig, axes = plt.subplots(5, 5, figsize=(7, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_img[i], cmap="gray")
    ax.set_title(f"정답: {y_train_m[i]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
# 사람 눈에도 애매한 글씨가 보이나요? 모델도 바로 그런 글씨들을 헷갈려합니다
```

</details>

In [ ]:
# 코드 입력

import numpy as np

# 사전 배포된 mnist_data.npz를 노트북과 같은 폴더에 두고 실행하세요
# .npz는 Numpy 데이터의 압축파일
# 이미 x_train, x_test, y_train, y_test로 분할되어 있음
mnist = np.load("mnist_data.npz")










# 사람 눈에도 애매한 글씨가 보이나요? 모델도 바로 그런 글씨들을 헷갈려합니다


### 2) 모델 조립 — 치트시트 실전 적용

이미지(28×28)는 펼치면 784개의 숫자입니다. 입력 784, 출력은 "10개 클래스" → 치트시트대로 **뉴런 10개 + softmax + sparse_categorical_crossentropy**.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 이미지 → 신경망 입력: 28×28을 784로 펼치고, 픽셀값(0~255)을 0~1로 스케일링
# 수업 속도를 위해 1만 장만 사용 (전체 6만 장으로 바꿔도 됩니다)
X_mnist = x_train_img[:10000].reshape(-1, 784) / 255.0
y_mnist = y_train_m[:10000]
X_mnist_test = x_test_img[:2000].reshape(-1, 784) / 255.0
y_mnist_test = y_test_m[:2000]

digit_model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),          # 치트시트: 다중분류 → 뉴런 10개 + softmax
])
digit_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",           # 치트시트: softmax와 세트
    metrics=["accuracy"],
)

digit_model.fit(X_mnist, y_mnist, epochs=15, batch_size=64, validation_split=0.2, verbose=1)
test_acc = digit_model.evaluate(X_mnist_test, y_mnist_test, verbose=0)[1]
print(f"\n테스트 정확도: {test_acc:.4f}")
```

</details>

In [ ]:
# 이미지 → 신경망 입력: 28×28을 784로 펼치고, 픽셀값(0~255)을 0~1로 스케일링
# 28×28 행렬(Matrix)을 길이 784의 벡터(Vector)
# 784차원 벡터(784-dimensional vector)

# 수업 속도를 위해 1만 장만 사용 (전체 6만 장으로 바꿔도 됩니다)
# reshape(-1, 784) : 벡터로






digit_model.fit(X_mnist, y_mnist, epochs=15, batch_size=64, validation_split=0.2, verbose=1)
test_acc = digit_model.evaluate(X_mnist_test, y_mnist_test, verbose=0)[1]
print(f"\n테스트 정확도: {test_acc:.4f}")


### 3) 모델이 헷갈린 글씨 구경하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
pred_probs = digit_model.predict(X_mnist_test, verbose=0)
pred_labels = pred_probs.argmax(axis=1)
wrong_idx = np.where(pred_labels != y_mnist_test)[0][:6]

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, idx in zip(axes, wrong_idx):
    ax.imshow(x_test_img[idx], cmap="gray")
    ax.set_title(f"정답 {y_mnist_test[idx]} → 예측 {pred_labels[idx]}", fontsize=9)
    ax.axis("off")
plt.show()
# 여러분이 보기엔 몇 번이 제일 억울(?)한 오답인가요? Zoom 채팅으로 알려주세요
```

</details>

In [ ]:
# 예측 확률 구하기




# 가장 높은 확률의 숫자 선택하기 (즉 어떤숫자로 예측했는가?)
# 0~9까지 0 (0.05), 1 (0.1), 2 (0.8) ...





# 틀린 위치만 찾기 (np.where의 조건이 True만 반환)




fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))

for ax, idx in zip(axes, wrong_idx):
    ax.imshow(x_test_img[idx], cmap="gray")
    ax.set_title(f"true {y_mnist_test[idx]} → pred {pred_labels[idx]}", fontsize=9)
    ax.axis("off")

plt.show()


### 4) 🎨 오늘의 하이라이트 — 내가 그린 숫자를 내 모델이 읽는다

아래 셀을 실행하면 그림판이 나타납니다 (Colab 전용). **마우스로 숫자 하나(0~9)를 크고 두껍게** 그린 뒤 [완료]를 누르세요.

> 💡 Colab이 아닌 환경에서는 그림판이 동작하지 않습니다 — 그 경우 이 셀을 건너뛰면 배포된 `sample_digit.png`로 자동 진행됩니다. 컴퓨터 그림판 앱으로 그린 PNG를 `my_digit.png` 이름으로 업로드해도 됩니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 그림판 열기 (Colab 전용 — 다른 환경에서는 건너뛰세요)
try:
    from google.colab.output import eval_js
    from IPython.display import HTML, display
    from base64 import b64decode

    canvas_html = '''
    <canvas width=280 height=280 style="border:2px solid #888; background:#ffffff; cursor:crosshair"></canvas>
    <br><button id="finish">완료</button> <button id="clear">지우기</button>
    <script>
    var canvas = document.querySelector("canvas");
    var ctx = canvas.getContext("2d");
    ctx.fillStyle = "#ffffff"; ctx.fillRect(0, 0, 280, 280);
    ctx.strokeStyle = "#000000"; ctx.lineWidth = 18; ctx.lineCap = "round";
    var drawing = false; var pos = {x: 0, y: 0};
    canvas.addEventListener("mousedown", e => { drawing = true; pos = {x: e.offsetX, y: e.offsetY}; });
    canvas.addEventListener("mouseup", () => drawing = false);
    canvas.addEventListener("mousemove", e => {
      if (!drawing) return;
      ctx.beginPath(); ctx.moveTo(pos.x, pos.y); ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke();
      pos = {x: e.offsetX, y: e.offsetY};
    });
    document.querySelector("#clear").onclick = () => { ctx.fillRect(0, 0, 280, 280); };
    var data = new Promise(resolve => {
      document.querySelector("#finish").onclick = () => resolve(canvas.toDataURL("image/png"));
    });
    </script>
    '''
    display(HTML(canvas_html))
    data_url = eval_js("data")
    with open("my_digit.png", "wb") as f:
        f.write(b64decode(data_url.split(",")[1]))
    print("my_digit.png 저장 완료")
except ImportError:
    print("Colab 환경이 아닙니다 — 배포된 sample_digit.png로 진행합니다")
```

</details>

In [ ]:
# 그림판 열기 (Colab 전용 — 다른 환경에서는 건너뛰세요)
try:
    from google.colab.output import eval_js
    from IPython.display import HTML, display
    from base64 import b64decode

    canvas_html = '''
    <canvas width=280 height=280 style="border:2px solid #888; background:#ffffff; cursor:crosshair"></canvas>
    <br><button id="finish">완료</button> <button id="clear">지우기</button>
    <script>
    var canvas = document.querySelector("canvas");
    var ctx = canvas.getContext("2d");
    ctx.fillStyle = "#ffffff"; ctx.fillRect(0, 0, 280, 280);
    ctx.strokeStyle = "#000000"; ctx.lineWidth = 18; ctx.lineCap = "round";
    var drawing = false; var pos = {x: 0, y: 0};
    canvas.addEventListener("mousedown", e => { drawing = true; pos = {x: e.offsetX, y: e.offsetY}; });
    canvas.addEventListener("mouseup", () => drawing = false);
    canvas.addEventListener("mousemove", e => {
      if (!drawing) return;
      ctx.beginPath(); ctx.moveTo(pos.x, pos.y); ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke();
      pos = {x: e.offsetX, y: e.offsetY};
    });
    document.querySelector("#clear").onclick = () => { ctx.fillRect(0, 0, 280, 280); };
    var data = new Promise(resolve => {
      document.querySelector("#finish").onclick = () => resolve(canvas.toDataURL("image/png"));
    });
    </script>
    '''
    display(HTML(canvas_html))
    data_url = eval_js("data")
    with open("my_digit.png", "wb") as f:
        f.write(b64decode(data_url.split(",")[1]))
    print("my_digit.png 저장 완료")
except ImportError:
    print("Colab 환경이 아닙니다 — 배포된 sample_digit.png로 진행합니다")


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 내가 그린 숫자를 모델 입력 형태(28×28)로 변환하고 예측
import os
from PIL import Image

path = "my_digit.png" if os.path.exists("my_digit.png") else "sample_digit.png"

img = Image.open(path).convert("RGBA")
bg = Image.new("RGBA", img.size, (255, 255, 255, 255))     # 투명 배경 대비 흰 배경 합성
img = Image.alpha_composite(bg, img).convert("L").resize((28, 28))

arr = np.array(img).astype("float32")
if arr.mean() > 127:          # 흰 배경·검은 글씨라면 반전 (모델은 검은 배경·흰 글씨로 학습됨)
    arr = 255 - arr
arr = arr / 255.0

probs = digit_model.predict(arr.reshape(1, 784), verbose=0)[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(arr, cmap="gray")
axes[0].set_title(f"모델이 본 내 글씨 → 예측: {probs.argmax()}")
axes[0].axis("off")
axes[1].bar(range(10), probs)
axes[1].set_xticks(range(10))
axes[1].set_title("숫자별 확신도 (softmax 출력)")
plt.tight_layout()
plt.show()
print(f"모델의 답: {probs.argmax()} (확신도 {probs.max()*100:.1f}%)")
```

</details>

In [ ]:
# 내가 그린 숫자를 모델 입력 형태(28×28)로 변환하고 예측

import os
from PIL import Image

path = "my_digit.png" if os.path.exists("my_digit.png") else "sample_digit.png"

img = Image.open(path).convert("RGBA")
bg = Image.new("RGBA", img.size, (255, 255, 255, 255))     # 투명 배경 대비 흰 배경 합성
img = Image.alpha_composite(bg, img).convert("L").resize((28, 28))

arr = np.array(img).astype("float32")
if arr.mean() > 127:          # 흰 배경·검은 글씨라면 반전 (모델은 검은 배경·흰 글씨로 학습됨)
    arr = 255 - arr
arr = arr / 255.0

probs = digit_model.predict(arr.reshape(1, 784), verbose=0)[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(arr, cmap="gray")
axes[0].set_title(f"모델이 본 내 글씨 → 예측: {probs.argmax()}")
axes[0].axis("off")
axes[1].bar(range(10), probs)
axes[1].set_xticks(range(10))
axes[1].set_title("숫자별 확신도 (softmax 출력)")
plt.tight_layout()
plt.show()
print(f"모델의 답: {probs.argmax()} (확신도 {probs.max()*100:.1f}%)")


> ✏️ **직접 해보기**: 그림판 셀부터 다시 실행해 다른 숫자로 도전해보세요. 일부러 흘려 쓰거나 작게 써서 모델을 속여보는 것도 재미있습니다 — softmax 확신도가 어떻게 낮아지는지 관찰하세요. 성공/실패 스크린샷은 Zoom 채팅에 자유롭게 공유해주세요!

---

## 🚀 실무에서는?

> 🚀 방금 여러분이 만든 "학습된 모델 + 실시간 입력 전처리 + 예측"이 바로 AI 서비스의 최소 단위입니다. 실무에서는 여기에 웹 API를 붙이면 그대로 서비스가 됩니다. 그리고 입력 전처리(크기·색·스케일)를 **학습 데이터와 똑같이** 맞추는 것이 서비스 품질의 절반입니다 — 방금 색 반전 한 줄이 그 예입니다.

---

## 🧩 Check Point

**Q1.** softmax 출력 10개를 다 더하면? ① 10 ② 1 ③ 정해지지 않음
<details><summary>정답</summary>② 1 (클래스별 확률이라서 — 확신도 막대그래프의 합이 항상 1)</details>

**Q2.** 내 손글씨를 예측할 때 색 반전(255-arr)이 필요했던 이유는? ① 모델이 흑백만 지원해서 ② 학습 데이터가 "검은 배경·흰 글씨"라서 입력도 똑같이 맞춰야 하므로 ③ 파일 크기를 줄이려고
<details><summary>정답</summary>② 학습 데이터와 입력 형태를 똑같이 맞춰야 하므로</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. (자습 시간에 진행합니다.)

In [ ]:
# TODO 1 (확신도 낮은 글씨): 테스트 데이터에서 모델의 최대 확신도가 가장 낮은(제일 헷갈려한) 이미지 5장을 찾아 시각화하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
max_conf = pred_probs.max(axis=1)
uncertain_idx = max_conf.argsort()[:5]
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for ax, idx in zip(axes, uncertain_idx):
    ax.imshow(x_test_img[idx], cmap="gray")
    ax.set_title(f"예측 {pred_labels[idx]} ({max_conf[idx]*100:.0f}%)", fontsize=9)
    ax.axis("off")
plt.show()
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (숫자별 정확도): 0~9 각 숫자별 정확도를 계산해, 모델이 어떤 숫자를 제일 어려워하는지 찾아보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for d in range(10):
    mask = y_mnist_test == d
    acc_d = (pred_labels[mask] == d).mean()
    print(f"숫자 {d}: 정확도 {acc_d:.3f}")
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

> 📝 자습 시간에 진행합니다.

In [ ]:
# 과제 1: 학습 데이터를 1만 장에서 3만 장으로 늘려 다시 학습시키고 테스트 정확도 변화를 확인하세요
#         ("데이터가 많을수록 얼마나 좋아지는가"를 숫자로 체감하기)

# 과제 2: 0~9를 전부 그림판(또는 그림 앱)으로 직접 그려 예측시키고, 몇 개나 맞히는지 "나만의 손글씨 정확도"를 계산하세요


In [ ]:
# 과제 1 답안





In [ ]:
# 과제 2 예시 정답 (파일명은 각자 저장한 이름으로)




---

## 📌 핵심 정리

- 다중분류 = 치트시트 조합: 뉴런 N개 + softmax + sparse_categorical_crossentropy
- 이미지는 펼치면(784) 그냥 숫자 데이터 — 오전과 같은 파이프라인이 그대로 작동
- 실시간 입력은 학습 데이터와 전처리를 똑같이 맞추는 것이 핵심 (크기·색 반전·스케일)
- "학습된 모델 + 입력 전처리 + 예측" = AI 서비스의 최소 단위

---

# 📘 CH14-05. 실무 마감 — 퇴근을 지켜주는 콜백, 그리고 최종 대결

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 과대적합을 눈으로 목격 → EarlyStopping(자동 멈춤) → ModelCheckpoint(최고의 순간 저장) → ⚔️ 최종 대결: 트리 vs 신경망
> 🔧 **실습파일**: 필요 (1교시 데이터 이어서 사용)
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★★☆

---

## 🤔 먼저 생각해보기

> **퇴근 직전, 사수의 마지막 조언 — "학습을 200 epoch 걸어놓고 퇴근했는데, 아침에 와보니 100 epoch부터는 오히려 나빠지고 있었다면? 시간도 버리고 모델도 버린 겁니다. 실무자는 학습을 '감시'하는 장치를 반드시 답니다. 그게 콜백이에요. 그리고 마지막으로 — 4일간 배운 트리 모델과 오늘의 신경망, 정면 대결 한번 봐야죠."**

1교시에 "훈련 성적은 오르는데 모의고사 성적이 멈추면?"이라고 예고했던 그 현상 — **과대적합**을 이제 눈으로 직접 목격하고, 자동으로 막는 장치를 답니다.

---

## 📖 실험

### 실험 A — 과대적합 목격하기: 일부러 200 epoch 방치

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 콜백 없이 200 epoch을 "방치" — 훈련 곡선과 모의고사(검증) 곡선이 어떻게 갈라지는지 관찰
model_over = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_over.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_over = model_over.fit(X_train_scaled, y_train, epochs=200, validation_split=0.2, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(history_over.history["loss"], label="훈련 loss (계속 내려감)")
plt.plot(history_over.history["val_loss"], label="검증 loss (어느 순간 반등!)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("과대적합의 현장 — 두 곡선이 갈라지는 순간")
plt.legend()
plt.show()
```

</details>

In [ ]:
# 콜백 없이 200 epoch을 "방치" — 훈련 곡선과 모의고사(검증) 곡선이 어떻게 갈라지는지 관찰

# 모델 생성







model_over.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_over = model_over.fit(X_train_scaled, y_train, epochs=200, validation_split=0.2, verbose=0)

plt.figure(figsize=(8, 4))

plt.plot(history_over.history["loss"], label="훈련 loss (계속 내려감)")
plt.plot(history_over.history["val_loss"], label="검증 loss (어느 순간 반등!)")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("과대적합의 현장 — 두 곡선이 갈라지는 순간")

plt.legend()
plt.show()


> 💡 훈련 loss는 0을 향해 계속 떨어지는데, 검증 loss는 어느 지점부터 **다시 올라갑니다.** 그 지점부터 모델은 "공부"가 아니라 "기출문제 암기"를 하고 있는 겁니다. 갈라지기 시작한 그 지점이 멈췄어야 할 순간 — 사람이 지켜볼 수 없으니, 자동 장치를 답니다.

### 실험 B — EarlyStopping: 알아서 멈추는 장치

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",           # 검증 loss를 감시
    patience=10,                    # 10 epoch 연속 개선이 없으면 중단
    restore_best_weights=True       # 멈출 때 "가장 좋았던 순간"의 가중치로 복원
)

model_es = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_es.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_es = model_es.fit(X_train_scaled, y_train, epochs=200,
                          validation_split=0.2, callbacks=[early_stop], verbose=0)

print(f"200 epoch 걸어뒀지만 → 실제 학습: {len(history_es.history['loss'])} epoch에서 자동 중단")
print("방치 모델 테스트 정확도:", round(model_over.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))
print("자동중단 모델 테스트 정확도:", round(model_es.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))
```

</details>

In [ ]:
# 코드 입력
from tensorflow.keras.callbacks import EarlyStopping

model_es = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model_es.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# EarlyStopping






print(f"200 epoch 걸어뒀지만 → 실제 학습: {len(history_es.history['loss'])} epoch에서 자동 중단")
print("방치 모델 테스트 정확도:", round(model_over.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))
print("자동중단 모델 테스트 정확도:", round(model_es.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))


### 실험 C — ModelCheckpoint: 최고의 순간을 파일로

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from tensorflow.keras.callbacks import ModelCheckpoint

# 실무 표준 패턴: EarlyStopping + ModelCheckpoint를 함께
model_final = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_final.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model_final.fit(
    X_train_scaled, y_train, epochs=200, validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True),  # 최고 시점만 저장
    ],
    verbose=0,
)

# 저장된 최고 모델을 다시 불러와 확인 — 내일 다시 켜도, 서버에 배포해도 이 파일이면 됩니다
best = keras.models.load_model("best_model.keras")
print("파일에서 복원한 모델 테스트 정확도:", round(best.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))
```

</details>

In [ ]:
# 코드 입력

from tensorflow.keras.callbacks import ModelCheckpoint

# 실무 표준 패턴: EarlyStopping + ModelCheckpoint를 함께

model_final = keras.Sequential([
    keras.Input(shape=(44,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model_final.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# ModelCheckpoint







# 저장된 최고 모델을 다시 불러와 확인 — 내일 다시 켜도, 서버에 배포해도 이 파일이면 됩니다
best = keras.models.load_model("best_model.keras")
print("파일에서 복원한 모델 테스트 정확도:", round(best.evaluate(X_test_scaled, y_test, verbose=0)[1], 4))


> 🚀 **실무에서는?** 실무 딥러닝에서 이 두 콜백 없이 학습시키는 경우는 거의 없습니다 — "퇴근 전에 걸어두고, 아침에 최고 성능 파일을 받는" 것이 표준 루틴입니다. `best_model.keras` 파일이 곧 배포되는 산출물입니다.

### ⚔️ 최종 대결 — 4일의 마무리: 트리(XGBoost) vs 신경망

"딥러닝이 최신인데 무조건 더 좋은 것 아닌가요?" — 실무에서 정말 자주 받는 질문입니다. 같은 퇴사 예측 데이터에서 정면 대결로 확인합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

# 선수 1: XGBoost (트리 기반 — 스케일링 불필요)
xgb = XGBClassifier(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# 선수 2: 오늘의 신경망 (콜백 포함 정석 학습을 마친 model_final)
nn_pred = (model_final.predict(X_test_scaled, verbose=0) >= 0.5).astype(int)

print(pd.DataFrame({
    "모델": ["XGBoost (트리)", "신경망 (MLP)"],
    "Accuracy": [round(accuracy_score(y_test, xgb_pred), 4), round(accuracy_score(y_test, nn_pred), 4)],
    "F1": [round(f1_score(y_test, xgb_pred), 4), round(f1_score(y_test, nn_pred), 4)],
}))
```

</details>

In [ ]:
# 코드 입력

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

# 선수 1: XGBoost (트리 기반 — 스케일링 불필요)





# 선수 2: 오늘의 신경망 (콜백 포함 정석 학습을 마친 model_final)





print(pd.DataFrame({
    "모델": ["XGBoost (트리)", "신경망 (MLP)"],
    "Accuracy": [round(accuracy_score(y_test, xgb_pred), 4), round(accuracy_score(y_test, nn_pred), 4)],
    "F1": [round(f1_score(y_test, xgb_pred), 4), round(f1_score(y_test, nn_pred), 4)],
}))


> 💡 표(정형) 데이터에서는 트리 기반 모델이 신경망과 **대등하거나 더 나은 경우가 많습니다** — 이것이 실무의 상식입니다. 그럼 딥러닝은 언제 압도적일까요? 바로 오늘 4교시에서 본 **이미지**, 그리고 음성·텍스트처럼 사람이 특성을 직접 뽑기 어려운 데이터입니다. **"문제에 맞는 도구를 고르는 것"이 엔지니어의 실력** — 내일 PyTorch에서 이미지 문제를 본격적으로 다루는 이유입니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ EarlyStopping의 `monitor`를 `val_loss`가 아닌 `loss`(훈련 loss)로 지정하면 과대적합을 전혀 못 막습니다 — 감시 대상은 항상 "모의고사 성적".

> ⚠️ `restore_best_weights=True`를 빼먹으면 "멈춘 시점"의 가중치가 남습니다 — 최고 시점이 아닐 수 있습니다.

> ⚠️ patience를 너무 작게(1~2) 잡으면 잠깐 주춤한 것뿐인데 너무 일찍 멈춰버립니다.

---

## 🧩 Check Point

**Q1.** 실험 A 그래프에서 "과대적합 시작"을 알리는 신호는? ① 훈련 loss가 0이 됨 ② 검증 loss가 다시 올라가기 시작함 ③ epoch이 100을 넘음
<details><summary>정답</summary>② 검증 loss의 반등</details>

**Q2.** "가장 좋았던 순간의 모델을 파일로 저장"하는 콜백은? ① EarlyStopping ② ModelCheckpoint
<details><summary>정답</summary>② ModelCheckpoint</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. (자습 시간에 진행합니다.)

In [ ]:
# TODO 1 (patience 실험): patience를 3, 10, 20으로 바꿔가며 실제 학습된 epoch 수가 어떻게 달라지는지 비교하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for p in [3, 10, 20]:
    m = keras.Sequential([
        keras.Input(shape=(44,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train_scaled, y_train, epochs=200, validation_split=0.2,
              callbacks=[EarlyStopping(monitor="val_loss", patience=p, restore_best_weights=True)], verbose=0)
    print(f"patience={p}: {len(h.history['loss'])} epoch에서 중단")
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 2 (MNIST에 콜백 적용): 4교시 손글씨 모델에 두 콜백을 함께 적용해 학습시키고, 저장된 최적 모델의 테스트 정확도를 확인하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
m = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(X_mnist, y_mnist, epochs=100, batch_size=64, validation_split=0.2,
      callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                 ModelCheckpoint("mnist_best.keras", monitor="val_accuracy", save_best_only=True)],
      verbose=0)
best_mnist = keras.models.load_model("mnist_best.keras")
print(best_mnist.evaluate(X_mnist_test, y_mnist_test, verbose=0))
```

</details>

In [ ]:
# 코드 입력





---

## 📝 실습 과제

> 📝 자습 시간에 진행합니다.

In [ ]:
# 과제 1: 실험 A의 200 epoch 곡선에서 검증 loss가 가장 낮았던 epoch이 몇 번째였는지 코드로 찾아보세요
#         (힌트: np.argmin) — EarlyStopping이 복원해주는 "최고의 순간"이 바로 그 지점입니다

# 과제 2: 최종 대결에 로지스틱 회귀도 참전시켜 3파전 표를 만들어보세요
#         (스케일링된 데이터 사용, max_iter=1000) — 가장 단순한 모델은 얼마나 따라올까요?


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
best_epoch = np.argmin(history_over.history["val_loss"]) + 1
print(f"검증 loss 최저점: {best_epoch}번째 epoch (그 뒤 {200 - best_epoch} epoch은 낭비 + 과대적합)")
```

</details>

In [ ]:
# 과제 1 답안





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

print(pd.DataFrame({
    "모델": ["Logistic (가장 단순)", "XGBoost (트리)", "신경망 (MLP)"],
    "Accuracy": [round(accuracy_score(y_test, lr_pred), 4),
                 round(accuracy_score(y_test, xgb_pred), 4),
                 round(accuracy_score(y_test, nn_pred), 4)],
    "F1": [round(f1_score(y_test, lr_pred), 4),
           round(f1_score(y_test, xgb_pred), 4),
           round(f1_score(y_test, nn_pred), 4)],
}))
```

</details>

In [ ]:
# 과제 2 답안






---

## 📌 핵심 정리

- 과대적합 = 훈련 loss와 검증 loss가 갈라지는 현상 — 학습 곡선으로 눈으로 진단
- EarlyStopping(자동 멈춤) + ModelCheckpoint(최고 시점 저장) = 실무 표준 세트
- 감시 대상은 항상 val_loss, restore_best_weights=True 필수
- 정형 데이터는 트리가 여전히 강하고, 딥러닝의 주 무대는 이미지·음성·텍스트 — 도구 선택이 실력

---

# 🎁 CH14-B (보너스). 자유 실험실 — 나만의 최고 기록 도전

━━━━━━━━━━━━━━━━

> ⏱️ **예상 소요시간**: 자유
> ⏭️ 진도 상황에 따라 건너뛸 수 있는 보너스이며, 정규 4시간 개념 진도에 포함되지 않습니다.

오늘 배운 부품(층 수, 뉴런 수, 활성화, 학습률, epoch, batch_size, 콜백)은 전부 여러분이 자유롭게 조합할 수 있는 레버입니다. 아래 템플릿으로 **손글씨(MNIST) 테스트 정확도 개인 최고 기록**에 도전해보세요 — 정답은 없고, 레버를 당길 때마다 숫자가 어떻게 변하는지 관찰하는 것 자체가 실무 감각입니다.

> 💡 힌트: 데이터 늘리기(1만→3만 장), 층 넓히기(256 뉴런), 콜백으로 epoch 넉넉히. 다만 "무조건 크게"가 항상 이기지는 않습니다 — 2교시 과제에서 확인했듯이요.

In [ ]:
# 코드 입력

# ✏️ 자유 실험 템플릿 — 레버를 마음껏 조정해보세요
my_model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),      # ← 뉴런 수·층 수·활성화 조정
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
my_model.compile(
    optimizer="adam",                           # ← 옵티마이저·학습률 조정
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
my_model.fit(
    X_mnist, y_mnist,                           # ← X_30k, y_30k로 데이터 늘리기 가능 (4교시 과제 참고)
    epochs=15, batch_size=64,                    # ← epoch·배치 조정 (콜백 달면 epoch 넉넉히)
    validation_split=0.2, verbose=0,
)
print("나의 기록:", round(my_model.evaluate(X_mnist_test, y_mnist_test, verbose=0)[1], 4))


---
## 🎉 수업 완료!

30분 만의 첫 신경망 → 활성화·학습 엔진 해부 실험 → 손글씨 인식 프로젝트 → 콜백과 최종 대결까지, 딥러닝의 전 과정을 "직접 만들고 부수면서" 완주했습니다. 원리·수식의 정리는 개념 교안(장표)을, 심화(Batch Normalization & Dropout)는 별도 md 문서를 참고해주세요.

내일 **CH15(PyTorch 입문)**에서는 오늘 Keras가 자동으로 해준 학습 과정을 밑바닥부터 직접 조립합니다 — 오늘 부숴본 부품들이 전부 다시 등장합니다!